In [1]:
"""
PROCESS OD DATASET (ALREADY DOWNLOADED)
============================================
1. Load existing file
2. Fix missing columns (n_boardings, n_alighting)
3. Feature engineering
4. Save processed datasets
"""

import pandas as pd
import numpy as np
import os
import calendar
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# ============================================
# CELL 1: CONFIGURATION
# ============================================

# ⭐ PATH TO YOUR EXISTING FILE
FILE_PATH = "/content/drive/MyDrive/Occupancy_capstone/Dataset/sunt_od_2024_03_march.parquet"  # Change this to your file path

# General settings
YEAR = 2024
MONTH = 3
BUS_CAPACITY = 80  # passengers

# Feature engineering settings
OCCUPANCY_METHOD = 'categorical'  # 'categorical', 'percentage', 'binary'
INCLUDE_LAGS = True
INCLUDE_AGGREGATIONS = False  # ⚠️ Disabled to save RAM
N_LAGS = 2  # ⚠️ Reduced from 3 to 2

# ⭐ MEMORY MODE
MEMORY_MODE = 'full'  # Options: 'minimal', 'medium', 'full'

# ⭐⭐ SAMPLING - CRITICAL FOR COLAB FREE ⭐⭐
USE_SAMPLING = True  # Set to True to reduce dataset size
SAMPLE_FRACTION = 0.5
# Recommendations:
# - 0.1 (10%) = ~2M records, ~1 GB RAM ✅ Safest
# - 0.3 (30%) = ~6M records, ~2 GB RAM ✅ Recommended
# - 0.5 (50%) = ~10M records, ~4 GB RAM ⚠️ Might crash
# - 1.0 (100%) = ~19M records, ~8 GB RAM ❌ Will crash

# Save settings
SAVE_RESULTS = True
SAVE_FORMAT = 'parquet'  # 'parquet' or 'csv'

print("=" * 70)
print("⚙️  CONFIGURATION")
print("=" * 70)
print(f"File to load: {FILE_PATH}")
print(f"Bus capacity: {BUS_CAPACITY} passengers")
print(f"Occupancy method: {OCCUPANCY_METHOD}")
print(f"Memory mode: {MEMORY_MODE.upper()}")
print(f"Lags: {N_LAGS if INCLUDE_LAGS else 0}")
print(f"Aggregations: {'Yes' if INCLUDE_AGGREGATIONS else 'No'}")
print(f"Save results: {'Yes' if SAVE_RESULTS else 'No'}")
print("=" * 70)


⚙️  CONFIGURATION
File to load: /content/drive/MyDrive/Occupancy_capstone/Dataset/sunt_od_2024_03_march.parquet
Bus capacity: 80 passengers
Occupancy method: categorical
Memory mode: FULL
Lags: 2
Aggregations: No
Save results: Yes


In [3]:

# ============================================
# CELL 2: LOAD EXISTING DATASET
# ============================================

def load_existing_dataset(filepath):
    """
    Load the SUNT OD dataset that you already downloaded
    """
    print("\n" + "=" * 70)
    print("📂 LOADING EXISTING DATASET")
    print("=" * 70)

    # Check if file exists
    if not os.path.exists(filepath):
        raise FileNotFoundError(
            f"❌ File not found: {filepath}\n"
            f"   Please check the path and try again."
        )

    print(f"\n📥 Loading from: {filepath}")

    # Detect format and load
    if filepath.endswith('.parquet'):
        df = pd.read_parquet(filepath)
        print("   Format: Parquet")
    elif filepath.endswith('.csv'):
        df = pd.read_csv(filepath)
        print("   Format: CSV")
    else:
        raise ValueError(f"Unsupported format. Use .parquet or .csv")

    # File info
    file_size_mb = os.path.getsize(filepath) / 1024**2
    print(f"   File size: {file_size_mb:.2f} MB")

    # Dataset info
    print(f"\n✅ Dataset loaded successfully")
    print(f"   Shape: {df.shape}")
    print(f"   Records: {len(df):,}")
    print(f"   Columns: {len(df.columns)}")
    print(f"   Memory: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

    # Show columns
    print(f"\n📋 Available columns:")
    for i, col in enumerate(df.columns, 1):
        print(f"   {i:2d}. {col}")

    return df

# Load the dataset
df = load_existing_dataset(FILE_PATH)

# ============================================
# CELL 2B: APPLY SAMPLING (CRITICAL FOR COLAB)
# ============================================

if USE_SAMPLING:
    print("\n" + "=" * 70)
    print("✂️  APPLYING SAMPLING TO REDUCE MEMORY")
    print("=" * 70)

    n_before = len(df)
    memory_before = df.memory_usage(deep=True).sum() / 1024**2

    print(f"\nBefore sampling:")
    print(f"   Records: {n_before:,}")
    print(f"   Memory: {memory_before:.2f} MB")

    # Apply random sampling
    df = df.sample(frac=SAMPLE_FRACTION, random_state=42)
    df = df.reset_index(drop=True)

    # Force garbage collection to free memory
    import gc
    gc.collect()

    n_after = len(df)
    memory_after = df.memory_usage(deep=True).sum() / 1024**2

    print(f"\nAfter sampling ({SAMPLE_FRACTION*100:.0f}%):")
    print(f"   Records: {n_after:,}")
    print(f"   Memory: {memory_after:.2f} MB")
    print(f"   Reduction: {(1 - n_after/n_before)*100:.1f}%")

    print("\n✅ Sampling applied successfully")
else:
    print("\n⏭️  Sampling disabled (USE_SAMPLING=False)")




📂 LOADING EXISTING DATASET

📥 Loading from: /content/drive/MyDrive/Occupancy_capstone/Dataset/sunt_od_2024_03_march.parquet
   Format: Parquet
   File size: 346.40 MB

✅ Dataset loaded successfully
   Shape: (19513956, 16)
   Records: 19,513,956
   Columns: 16
   Memory: 5349.80 MB

📋 Available columns:
    1. route_short_name
    2. register_code
    3. direction_id
    4. pt_sequence
    5. stop_id
    6. vehicle
    7. trip_number
    8. trip_id
    9. start_trip
   10. end_trip
   11. stop_time
   12. n-boardings
   13. n-alighting
   14. lag_loading
   15. balance
   16. loading

✂️  APPLYING SAMPLING TO REDUCE MEMORY

Before sampling:
   Records: 19,513,956
   Memory: 5349.80 MB

After sampling (50%):
   Records: 9,756,978
   Memory: 2674.89 MB
   Reduction: 50.0%

✅ Sampling applied successfully


In [4]:
# ============================================
# CELL 3: FIX MISSING COLUMNS
# ============================================

def fix_missing_columns(df):
    """
    Create n_boardings, n_alighting, and gps_datetime if missing
    """
    print("\n" + "=" * 70)
    print("🔧 FIXING MISSING COLUMNS")
    print("=" * 70)

    df = df.copy()

    # Check which columns are missing
    critical_columns = ['loading', 'n_boardings', 'n_alighting', 'gps_datetime']
    present = [c for c in critical_columns if c in df.columns]
    missing = [c for c in critical_columns if c not in df.columns]

    print(f"\n📊 Critical columns status:")
    for col in critical_columns:
        status = "✅" if col in df.columns else "❌"
        print(f"   {status} {col}")

    if not missing:
        print("\n✅ All critical columns already present")
        return df

    print(f"\n⚠️  Missing {len(missing)} columns: {missing}")
    print("   Creating them now...")

    # 1. Create gps_datetime if missing
    if 'gps_datetime' not in df.columns:
        print("\n1️⃣  Creating 'gps_datetime'...")
        if 'stop_time' in df.columns:
            df['gps_datetime'] = df['stop_time']
            print("   ✅ gps_datetime created (copy of stop_time)")
        elif 'start_trip' in df.columns:
            df['gps_datetime'] = df['start_trip']
            print("   ✅ gps_datetime created (copy of start_trip)")
        else:
            print("   ⚠️  No time column found to create gps_datetime")

    # Convert dates to datetime
    date_cols = ['gps_datetime', 'stop_time', 'start_trip']
    for col in date_cols:
        if col in df.columns:
            if not pd.api.types.is_datetime64_any_dtype(df[col]):
                df[col] = pd.to_datetime(df[col], errors='coerce')

    # 2. Verify 'loading' exists
    if 'loading' not in df.columns:
        raise ValueError(
            "❌ ERROR: Column 'loading' not found.\n"
            "   This column is critical to calculate n_boardings and n_alighting."
        )

    # 3. Create n_boardings and n_alighting if missing
    if 'n_boardings' not in df.columns or 'n_alighting' not in df.columns:
        print("\n2️⃣  Calculating 'n_boardings' and 'n_alighting'...")

        # Check grouping columns
        group_cols = ['route_short_name', 'direction_id', 'start_trip', 'pt_sequence']
        missing_group = [c for c in group_cols if c not in df.columns]

        if missing_group:
            print(f"   ⚠️  Missing grouping columns: {missing_group}")
            print("   Using simple method...")
            df = df.sort_index()
            df['loading_diff'] = df['loading'].diff()
        else:
            # Standard method
            print("   📊 Sorting data by trip and sequence...")
            df = df.sort_values(['route_short_name', 'direction_id', 'start_trip', 'pt_sequence'])
            df = df.reset_index(drop=True)

            print("   🔢 Calculating occupancy changes...")
            df['loading_diff'] = df.groupby(
                ['route_short_name', 'direction_id', 'start_trip']
            )['loading'].diff()

        # n_boardings = positive increases
        if 'n_boardings' not in df.columns:
            df['n_boardings'] = df['loading_diff'].clip(lower=0).fillna(0)
            print("   ✅ n_boardings created")

        # n_alighting = negative changes → positive
        if 'n_alighting' not in df.columns:
            df['n_alighting'] = (-df['loading_diff']).clip(lower=0).fillna(0)
            print("   ✅ n_alighting created")

        # First stop: all are boardings
        if not missing_group:
            first_stop = df.groupby(
                ['route_short_name', 'direction_id', 'start_trip']
            ).head(1).index
            df.loc[first_stop, 'n_boardings'] = df.loc[first_stop, 'loading']
            df.loc[first_stop, 'n_alighting'] = 0
            print("   ✅ First stop adjusted")

        # Clean temporary column
        df = df.drop(columns=['loading_diff'], errors='ignore')

    # 4. Create balance
    if 'balance' not in df.columns:
        df['balance'] = df['n_boardings'] - df['n_alighting']
        print("   ✅ balance created")

    # 5. Filter invalid values
    print("\n3️⃣  Filtering invalid values...")
    n_before = len(df)

    if 'loading' in df.columns:
        df = df[df['loading'] >= 0]
        n_after = len(df)
        if n_before != n_after:
            print(f"   ⚠️  Removed {n_before - n_after:,} records with negative loading")

    # 6. Validation
    print("\n" + "-" * 70)
    print("📊 VALIDATION:")
    print("-" * 70)

    if 'n_boardings' in df.columns:
        print(f"n_boardings:")
        print(f"   Mean: {df['n_boardings'].mean():.2f}")
        print(f"   Max: {df['n_boardings'].max():.0f}")
        print(f"   Total: {df['n_boardings'].sum():,.0f}")

    if 'n_alighting' in df.columns:
        print(f"n_alighting:")
        print(f"   Mean: {df['n_alighting'].mean():.2f}")
        print(f"   Max: {df['n_alighting'].max():.0f}")
        print(f"   Total: {df['n_alighting'].sum():,.0f}")

    print(f"\n✅ Columns fixed successfully")
    print(f"   Final shape: {df.shape}")

    return df

# Fix missing columns
df = fix_missing_columns(df)

# Free memory after each heavy operation
import gc
gc.collect()



🔧 FIXING MISSING COLUMNS

📊 Critical columns status:
   ✅ loading
   ❌ n_boardings
   ❌ n_alighting
   ❌ gps_datetime

⚠️  Missing 3 columns: ['n_boardings', 'n_alighting', 'gps_datetime']
   Creating them now...

1️⃣  Creating 'gps_datetime'...
   ✅ gps_datetime created (copy of stop_time)

2️⃣  Calculating 'n_boardings' and 'n_alighting'...
   📊 Sorting data by trip and sequence...
   🔢 Calculating occupancy changes...
   ✅ n_boardings created
   ✅ n_alighting created
   ✅ First stop adjusted

3️⃣  Filtering invalid values...

----------------------------------------------------------------------
📊 VALIDATION:
----------------------------------------------------------------------
n_boardings:
   Mean: 2.04
   Max: 407
   Total: 19,905,068
n_alighting:
   Mean: 1.09
   Max: 231
   Total: 10,646,603

✅ Columns fixed successfully
   Final shape: (9756978, 19)


17

In [5]:
# ============================================
# CELL 4: FEATURE ENGINEERING (MEMORY OPTIMIZED)
# ============================================

def apply_feature_engineering(df, capacity, n_lags, include_aggregations, mode='minimal'):
    """
    Apply feature engineering based on memory mode

    Parameters:
    -----------
    mode : str
        'minimal': Essential features only (low RAM usage)
        'medium': Important features (moderate RAM usage)
        'full': All features (high RAM usage)
    """
    print("\n" + "=" * 70)
    print("🎨 FEATURE ENGINEERING")
    print("=" * 70)
    print(f"Memory mode: {mode.upper()}")

    df = df.copy()
    features_created = []

    # ============================================
    # MINIMAL MODE: Essential features only
    # ============================================

    # 1. Basic temporal features (always)
    print("\n1️⃣  Basic temporal features...")
    if 'gps_datetime' in df.columns:
        df['hour'] = df['gps_datetime'].dt.hour
        df['day_of_week'] = df['gps_datetime'].dt.dayofweek
        df['is_weekend'] = (df['day_of_week'] >= 5).astype(int)
        df['is_rush_hour'] = (
            ((df['hour'] >= 7) & (df['hour'] <= 9)) |
            ((df['hour'] >= 17) & (df['hour'] <= 19))
        ).astype(int)

        features_created.extend(['hour', 'day_of_week', 'is_weekend', 'is_rush_hour'])
        print(f"   ✅ 4 basic temporal features created")
    else:
        print(f"   ⚠️  Skipped (gps_datetime missing)")

    # 2. Basic spatial feature (always)
    print("\n2️⃣  Basic spatial feature...")
    if 'pt_sequence' in df.columns and 'route_short_name' in df.columns:
        df['route_percentage'] = df.groupby(
            ['route_short_name', 'direction_id', 'start_trip']
        )['pt_sequence'].transform(lambda x: (x / x.max()) * 100)

        features_created.append('route_percentage')
        print(f"   ✅ 1 spatial feature created")
    else:
        print(f"   ⚠️  Skipped (missing required columns)")

    # 3. Minimal lag features (1-2 lags in minimal mode)
    if n_lags > 0 and 'loading' in df.columns:
        print(f"\n3️⃣  Lag features...")

        if mode == 'minimal':
            actual_lags = min(n_lags, 2)  # Maximum 2 lags
            print(f"   ⚠️  Limited to {actual_lags} lags (minimal mode)")
        else:
            actual_lags = n_lags

        if 'route_short_name' in df.columns:
            for lag in range(1, actual_lags + 1):
                col_name = f'loading_lag_{lag}'
                df[col_name] = df.groupby(
                    ['route_short_name', 'direction_id', 'start_trip']
                )['loading'].shift(lag)
                features_created.append(col_name)

            print(f"   ✅ {actual_lags} lag features created")
        else:
            print(f"   ⚠️  Skipped (missing grouping columns)")

    # ============================================
    # MEDIUM MODE: Add more features
    # ============================================

    if mode in ['medium', 'full']:

        # 4. Additional temporal features
        print("\n4️⃣  Additional temporal features...")
        if 'gps_datetime' in df.columns:
            df['month'] = df['gps_datetime'].dt.month

            # Time of day (efficient method)
            df['time_of_day'] = 'afternoon'
            df.loc[df['hour'] < 6, 'time_of_day'] = 'night'
            df.loc[(df['hour'] >= 6) & (df['hour'] < 12), 'time_of_day'] = 'morning'
            df.loc[df['hour'] >= 18, 'time_of_day'] = 'evening'

            features_created.extend(['month', 'time_of_day'])
            print(f"   ✅ 2 additional temporal features")

        # 5. Additional spatial feature
        print("\n5️⃣  Additional spatial feature...")
        if 'route_percentage' in df.columns:
            df['trip_stage'] = 'middle'
            df.loc[df['route_percentage'] <= 25, 'trip_stage'] = 'start'
            df.loc[df['route_percentage'] >= 75, 'trip_stage'] = 'end'

            features_created.append('trip_stage')
            print(f"   ✅ 1 additional spatial feature")

        # 6. Operational features
        print("\n6️⃣  Operational features...")
        if 'loading' in df.columns:
            df['passenger_density'] = df['loading'] / capacity
            features_created.append('passenger_density')
            print(f"   ✅ 1 operational feature")

        # 7. Moving average (only in medium/full mode)
        if 'loading' in df.columns and 'route_short_name' in df.columns:
            print("\n7️⃣  Moving average...")
            df['loading_ma_3'] = df.groupby(
                ['route_short_name', 'direction_id', 'start_trip']
            )['loading'].transform(lambda x: x.rolling(window=3, min_periods=1).mean())
            features_created.append('loading_ma_3')
            print(f"   ✅ Moving average created")

    # ============================================
    # FULL MODE: Add heavy features
    # ============================================

    if mode == 'full':

        # 8. Aggregation features (CONSUMES A LOT OF RAM)
        if include_aggregations and 'route_short_name' in df.columns:
            print("\n8️⃣  Aggregation features (heavy)...")

            # By route (only mean, not std to save memory)
            line_stats = df.groupby('route_short_name')['loading'].mean().reset_index()
            line_stats.columns = ['route_short_name', 'loading_mean_route']
            df = df.merge(line_stats, on='route_short_name', how='left')
            features_created.append('loading_mean_route')

            # By stop (only mean)
            if 'stop_id' in df.columns and 'n_boardings' in df.columns:
                stop_stats = df.groupby('stop_id')['n_boardings'].mean().reset_index()
                stop_stats.columns = ['stop_id', 'boardings_mean_stop']
                df = df.merge(stop_stats, on='stop_id', how='left')
                features_created.append('boardings_mean_stop')

            print(f"   ✅ 2 aggregation features created")
            print(f"   ⚠️  This significantly increases RAM usage")


    # ============================================
    # CLEAN OUTLIERS
    # ============================================
    print("\n🧹 Cleaning outliers...")
    n_before = len(df)

    # Remove impossible values (loading > 120)
    df = df[df['loading'] <= 120].copy()

    n_after = len(df)
    outliers_removed = n_before - n_after

    if outliers_removed > 0:
        print(f"   ⚠️  Removed {outliers_removed:,} outliers (loading > 120)")
        print(f"   ({outliers_removed/n_before*100:.2f}% of data)")
    else:
        print(f"   ✅ No outliers found")

    print(f"   Final records: {n_after:,}")


    # ============================================
    # CREATE TARGET VARIABLE (ALWAYS)
    # ============================================
    print(f"\n🎯 Creating target variable (occupancy_level - {OCCUPANCY_METHOD})...")
    if OCCUPANCY_METHOD == 'categorical':
        occupancy_pct = (df['loading'] / capacity) * 100

        # Adjusted thresholds for Paraguayan reality
        df['occupancy_level'] = 'medium'
        df.loc[occupancy_pct < 25, 'occupancy_level'] = 'low'        # 0-20 passengers
        df.loc[occupancy_pct >= 50, 'occupancy_level'] = 'high'      # 40-65 passengers
        df.loc[occupancy_pct >= 82, 'occupancy_level'] = 'very_high' # 66+ passengers

        print("\n📊 Threshold ranges (for capacity=80):")
        print(f"   low:       0-20  passengers (0-24%)")
        print(f"   medium:   21-39  passengers (25-49%)")
        print(f"   high:     40-65  passengers (50-81%)")
        print(f"   very_high: 66+   passengers (82%+)")

    elif OCCUPANCY_METHOD == 'percentage':
        df['occupancy_level'] = ((df['loading'] / capacity) * 100).clip(0, 150)
    else:  # binary
        threshold = 0.8 * capacity
        df['occupancy_level'] = (df['loading'] > threshold).astype(int)

    print(f"   ✅ occupancy_level created")

    # ============================================
    # CLEAN NaN VALUES (OPTIMIZED)
    # ============================================

    print("\n🧹 Cleaning missing values...")
    n_before = len(df)

    # Remove NaN in target
    df = df.dropna(subset=['occupancy_level'])

    # Remove NaN in lags
    if n_lags > 0:
        lag_cols = [f'loading_lag_{i}' for i in range(1, (n_lags if mode != 'minimal' else min(n_lags, 2)) + 1)]
        lag_cols_exist = [c for c in lag_cols if c in df.columns]
        if lag_cols_exist:
            df = df.dropna(subset=lag_cols_exist)

    n_after = len(df)

    if n_before != n_after:
        print(f"   ⚠️  Removed {n_before - n_after:,} records with NaN ({(n_before-n_after)/n_before*100:.2f}%)")

    # ============================================
    # OPTIMIZE MEMORY
    # ============================================

    print("\n💾 Optimizing memory usage...")

    # Convert int64 to int32 (saves 50% memory)
    for col in df.select_dtypes(include=['int64']).columns:
        df[col] = df[col].astype('int32')

    # Convert float64 to float32 (saves 50% memory)
    for col in df.select_dtypes(include=['float64']).columns:
        df[col] = df[col].astype('float32')

    # Convert object to category (saves ~70% memory)
    for col in df.select_dtypes(include=['object']).columns:
        if df[col].nunique() < len(df) * 0.5:
            df[col] = df[col].astype('category')

    print(f"   ✅ Data types optimized")

    # ============================================
    # SUMMARY
    # ============================================

    memory_mb = df.memory_usage(deep=True).sum() / 1024**2

    print("\n" + "-" * 70)
    print(f"✅ Feature engineering completed ({mode.upper()})")
    print(f"   Final shape: {df.shape}")
    print(f"   Features created: {len(features_created)}")
    print(f"   Total columns: {len(df.columns)}")
    print(f"   Memory usage: {memory_mb:.2f} MB")

    # Memory recommendations
    if memory_mb > 4000:
        print(f"\n⚠️  WARNING: High memory usage ({memory_mb:.0f} MB)")
        print(f"   Recommendation: Use 'minimal' or 'medium' mode")
    elif memory_mb > 2000:
        print(f"\n✅ Moderate memory usage ({memory_mb:.0f} MB)")
    else:
        print(f"\n✅ Low memory usage ({memory_mb:.0f} MB)")

    return df

# Apply feature engineering with configured mode
df = apply_feature_engineering(
    df,
    capacity=BUS_CAPACITY,
    n_lags=N_LAGS if INCLUDE_LAGS else 0,
    include_aggregations=INCLUDE_AGGREGATIONS,
    mode=MEMORY_MODE
)

# Free memory after feature engineering
import gc
gc.collect()




🎨 FEATURE ENGINEERING
Memory mode: FULL

1️⃣  Basic temporal features...
   ✅ 4 basic temporal features created

2️⃣  Basic spatial feature...
   ✅ 1 spatial feature created

3️⃣  Lag features...
   ✅ 2 lag features created

4️⃣  Additional temporal features...
   ✅ 2 additional temporal features

5️⃣  Additional spatial feature...
   ✅ 1 additional spatial feature

6️⃣  Operational features...
   ✅ 1 operational feature

7️⃣  Moving average...
   ✅ Moving average created

🧹 Cleaning outliers...
   ⚠️  Removed 5,459 outliers (loading > 120)
   (0.06% of data)
   Final records: 9,751,519

🎯 Creating target variable (occupancy_level - categorical)...

📊 Threshold ranges (for capacity=80):
   low:       0-20  passengers (0-24%)
   medium:   21-39  passengers (25-49%)
   high:     40-65  passengers (50-81%)
   very_high: 66+   passengers (82%+)
   ✅ occupancy_level created

🧹 Cleaning missing values...
   ⚠️  Removed 1,206,932 records with NaN (12.38%)

💾 Optimizing memory usage...
   ✅ D

27

In [6]:
# ============================================
# CELL 5: PREPARE FOR MODELING
# ============================================

def prepare_for_modeling(df):
    """
    Separate X and y, encode categorical variables
    """
    print("\n" + "=" * 70)
    print("🎯 PREPARING FOR MODELING")
    print("=" * 70)

    # Separate target
    y = df['occupancy_level'].copy()

    # Remove unnecessary columns
    cols_to_drop = [
        'occupancy_level',
        'gps_datetime', 'stop_time', 'start_trip',
        'cod_card', 'vehicle',
        # ⭐ FIX DATA LEAKAGE ⭐
        'loading',            # CRITICAL
        'balance',
        'passenger_density',
        'lag_loading',
        'loading_lag_1', 'loading_lag_2',
        'loading_ma_3',
        'n_alighting', 'n-alighting',
        'n_boardings', 'n-boardings',
        'stop_id', 'trip_id',
    ]
    cols_to_drop = [col for col in cols_to_drop if col in df.columns]

    X = df.drop(columns=cols_to_drop)

    # Encode categorical variables
    print("\n1️⃣  Encoding categorical variables...")
    categorical_cols = X.select_dtypes(include=['object', 'category']).columns

    from sklearn.preprocessing import LabelEncoder
    label_encoders = {}

    for col in categorical_cols:
        label_encoders[col] = LabelEncoder()
        X[col] = label_encoders[col].fit_transform(X[col].astype(str))

    print(f"   ✅ {len(categorical_cols)} columns encoded")

    # Handle infinity and NaN
    print("\n2️⃣  Cleaning special values...")
    X = X.replace([np.inf, -np.inf], np.nan)
    X = X.fillna(X.median())

    print("\n" + "-" * 70)
    print("📊 FINAL DATASET:")
    print("-" * 70)
    print(f"   X shape: {X.shape}")
    print(f"   y shape: {y.shape}")
    print(f"   Features: {len(X.columns)}")

    if OCCUPANCY_METHOD == 'categorical':
        print(f"\n   Target distribution:")
        dist = y.value_counts(normalize=True).sort_index()
        for category, prop in dist.items():
            print(f"      {category}: {prop*100:.2f}%")
    else:
        print(f"\n   Target statistics:")
        print(f"      Mean: {y.mean():.2f}")
        print(f"      Median: {y.median():.2f}")

    feature_names = X.columns.tolist()

    return X, y, feature_names, label_encoders

# Prepare for modeling
X, y, feature_names, label_encoders = prepare_for_modeling(df)


🎯 PREPARING FOR MODELING

1️⃣  Encoding categorical variables...
   ✅ 5 columns encoded

2️⃣  Cleaning special values...

----------------------------------------------------------------------
📊 FINAL DATASET:
----------------------------------------------------------------------
   X shape: (8544587, 14)
   y shape: (8544587,)
   Features: 14

   Target distribution:
      high: 11.67%
      low: 57.71%
      medium: 27.23%
      very_high: 3.39%


In [7]:
# ============================================
# CELL 6: SAVE RESULTS
# ============================================

def save_results(df, X, y, year, month, format='parquet'):
    """
    Save processed datasets to disk
    """
    import calendar
    import pickle

    if not SAVE_RESULTS:
        print("\n⏭️  Saving disabled (SAVE_RESULTS=False)")
        return

    print("\n" + "=" * 70)
    print("💾 SAVING RESULTS")
    print("=" * 70)

    month_name = calendar.month_name[month].lower()

    # 1. Save complete processed dataset
    filename_processed = f"sunt_od_processed_{year}_{month:02d}_{month_name}.{format}"
    print(f"\n1️⃣  Saving processed dataset...")
    print(f"   File: {filename_processed}")

    if format == 'parquet':
        df.to_parquet(filename_processed, index=False)
    else:
        df.to_csv(filename_processed, index=False)

    size = os.path.getsize(filename_processed) / 1024**2
    print(f"   ✅ Saved ({size:.2f} MB)")

    # 2. Save X (features)
    filename_X = f"sunt_X_{year}_{month:02d}_{month_name}.parquet"
    print(f"\n2️⃣  Saving features (X)...")
    print(f"   File: {filename_X}")
    X.to_parquet(filename_X, index=False)
    size_X = os.path.getsize(filename_X) / 1024**2
    print(f"   ✅ Saved ({size_X:.2f} MB)")

    # 3. Save y (target)
    filename_y = f"sunt_y_{year}_{month:02d}_{month_name}.pkl"
    print(f"\n3️⃣  Saving target (y)...")
    print(f"   File: {filename_y}")
    with open(filename_y, 'wb') as f:
        pickle.dump(y, f)
    size_y = os.path.getsize(filename_y) / 1024**2
    print(f"   ✅ Saved ({size_y:.2f} MB)")

    # Summary
    print("\n" + "-" * 70)
    print("✅ FILES SAVED:")
    print("-" * 70)
    print(f"1. {filename_processed} ({size:.2f} MB)")
    print(f"2. {filename_X} ({size_X:.2f} MB)")
    print(f"3. {filename_y} ({size_y:.2f} MB)")
    print(f"\nTotal: {size + size_X + size_y:.2f} MB")
    print(f"📁 Location: {os.path.abspath('.')}")

# Save results
save_results(df, X, y, YEAR, MONTH, SAVE_FORMAT)

# ============================================
# CELL 7: FINAL SUMMARY
# ============================================

print("\n" + "=" * 70)
print("🎉 PROCESSING COMPLETED SUCCESSFULLY")
print("=" * 70)

print("\n📊 SUMMARY:")
print("-" * 70)
print(f"Original dataset: {FILE_PATH}")
print(f"Records processed: {len(df):,}")
print(f"Features created: {len(X.columns)}")
print(f"Target: occupancy_level ({OCCUPANCY_METHOD})")
print(f"Memory mode: {MEMORY_MODE}")

print("\n⚠️  DATA LEAKAGE PREVENTION:")
print("   Removed: loading, balance, passenger_density")
print("   These columns were used to calculate the target")

if SAVE_RESULTS:
    import calendar
    month_name = calendar.month_name[MONTH].lower()
    print(f"\n📁 FILES READY FOR TRAINING:")
    print(f"   - sunt_X_{YEAR}_{MONTH:02d}_{month_name}.parquet")
    print(f"   - sunt_y_{YEAR}_{MONTH:02d}_{month_name}.pkl")

print("\n💡 NEXT STEPS:")
print("   1. Load X and y in training notebook")
print("   2. Train Random Forest / XGBoost model")
print("   3. Evaluate and optimize")
print("=" * 70)

print("\n✅ Variables available in memory:")
print("   - df: Complete DataFrame with features")
print("   - X: Features for modeling")
print("   - y: Target variable")
print("   - feature_names: List of feature names")


💾 SAVING RESULTS

1️⃣  Saving processed dataset...
   File: sunt_od_processed_2024_03_march.parquet
   ✅ Saved (463.63 MB)

2️⃣  Saving features (X)...
   File: sunt_X_2024_03_march.parquet
   ✅ Saved (26.32 MB)

3️⃣  Saving target (y)...
   File: sunt_y_2024_03_march.pkl
   ✅ Saved (138.53 MB)

----------------------------------------------------------------------
✅ FILES SAVED:
----------------------------------------------------------------------
1. sunt_od_processed_2024_03_march.parquet (463.63 MB)
2. sunt_X_2024_03_march.parquet (26.32 MB)
3. sunt_y_2024_03_march.pkl (138.53 MB)

Total: 628.48 MB
📁 Location: /content

🎉 PROCESSING COMPLETED SUCCESSFULLY

📊 SUMMARY:
----------------------------------------------------------------------
Original dataset: /content/drive/MyDrive/Occupancy_capstone/Dataset/sunt_od_2024_03_march.parquet
Records processed: 8,544,587
Features created: 14
Target: occupancy_level (categorical)
Memory mode: full

⚠️  DATA LEAKAGE PREVENTION:
   Removed: lo

In [8]:
print("=" * 70)
print("📋 TODAS LAS COLUMNAS DEL DATASET SUNT OD")
print("=" * 70)

print(f"\nTotal de columnas: {len(df.columns)}")
print("\n" + "-" * 70)

# Mostrar todas las columnas con su tipo de dato
for i, col in enumerate(df.columns, 1):
    dtype = df[col].dtype
    n_unique = df[col].nunique()
    print(f"{i:3d}. {col:30s} | dtype: {str(dtype):15s} | unique: {n_unique:>8,}")

print("-" * 70)

📋 TODAS LAS COLUMNAS DEL DATASET SUNT OD

Total de columnas: 32

----------------------------------------------------------------------
  1. route_short_name               | dtype: category        | unique:      397
  2. register_code                  | dtype: int32           | unique:      443
  3. direction_id                   | dtype: category        | unique:        2
  4. pt_sequence                    | dtype: int16           | unique:       89
  5. stop_id                        | dtype: int32           | unique:    2,802
  6. vehicle                        | dtype: category        | unique:    1,798
  7. trip_number                    | dtype: int32           | unique:       66
  8. trip_id                        | dtype: category        | unique:  202,860
  9. start_trip                     | dtype: datetime64[ns]  | unique:  511,336
 10. end_trip                       | dtype: category        | unique:  512,099
 11. stop_time                      | dtype: datetime64[ns]  | u